# 08 Analysis with ATAC Data

In [3]:
# Initialize
cd /home/dalbao/AlbaoRunx3Manuscript/cutnrun

docker_run() {
    docker run --rm -i \
        -u $(id -u):$(id -g) \
        -e MPLCONFIGDIR=/home/dalbao/.config/matplotlib \
        -v /tmp:/tmp \
        -v /home/dalbao:/home/dalbao \
        -v /etc/timezone:/etc/timezone:ro \
        -v /etc/localtime:/etc/localtime:ro \
        -w $(pwd) \
        "$@"
}

# Define software to use:
## deeptools for analysis and visualization of deep-sequencing data
deeptools() {
    docker_run quay.io/biocontainers/deeptools:3.5.6--pyhdfd78af_0 "$@"
}
deeptools plotHeatmap --version

# Define software to use:
## bedtools for bed file manipulation
bedtools() {
    docker_run staphb/bedtools:2.31.1 bedtools "$@"
}
bedtools --version

plotHeatmap 3.5.6
bedtools v2.31.1


In [11]:
# First make a associative array to store bigWig and peak locations for each sample
fpath="source_data/atac_bw/"

declare -A bigWigFiles
for group in GFPpos GFPneg CX3CR1pos; do
    for target in shCD19 shRunx3; do
        fn=${group}_${target}.bw
        bigWigFiles[${group}_${target}]="${fpath}${fn}"
    done
done


# Compute Matrix
mkdir -p 08_atac

deeptools computeMatrix reference-point \
    -S \
    ${bigWigFiles["GFPpos_shCD19"]} \
    ${bigWigFiles["GFPpos_shRunx3"]} \
    ${bigWigFiles["GFPneg_shCD19"]} \
    ${bigWigFiles["GFPneg_shRunx3"]} \
    ${bigWigFiles["CX3CR1pos_shCD19"]} \
    ${bigWigFiles["CX3CR1pos_shRunx3"]} \
    -R \
    "06_replicates/clusters_runx3/Cluster_1.bed" \
    "06_replicates/clusters_runx3/Cluster_2.bed" \
    "06_replicates/clusters_runx3/Cluster_3.bed" \
    "06_replicates/clusters_runx3/Cluster_4.bed" \
    "06_replicates/clusters_runx3/Cluster_5.bed" \
    "06_replicates/clusters_runx3/Cluster_6.bed" \
    "06_replicates/clusters_runx3/Cluster_7.bed" \
    "06_replicates/clusters_runx3/Cluster_8.bed" \
    --referencePoint center \
    -b 1500 -a 1500 \
    --numberOfProcessors 48 \
    --sortUsing mean \
    -out "08_atac/atac.matrix.gz"

deeptools plotHeatmap \
    -m "08_atac/atac.matrix.gz" \
    -out "08_atac/atac.matrix.pdf" \
    --sortUsing mean \
    --colorMap  RdYlBu_r RdYlBu_r\
                RdYlBu_r RdYlBu_r RdYlBu_r RdYlBu_r \
    --samplesLabel  "GFPpos_shCd19" "GFPpos_shRunx3" \
                    "GFPneg_shCd19" "GFPneg_shRunx3" "CX3CR1pos_shCd19" "CX3CR1pos_shRunx3" \
    --regionsLabel "C1" "C2" "C3" "C4" "C5" "C6" "C7" "C8"

In [12]:
for cluster in {1..8}; do
    deeptools computeMatrix reference-point \
        -S \
        ${bigWigFiles["GFPpos_shCD19"]} \
        ${bigWigFiles["GFPpos_shRunx3"]} \
        ${bigWigFiles["GFPneg_shCD19"]} \
        ${bigWigFiles["GFPneg_shRunx3"]} \
        ${bigWigFiles["CX3CR1pos_shCD19"]} \
        ${bigWigFiles["CX3CR1pos_shRunx3"]} \
        -R \
        "06_replicates/clusters_runx3/Cluster_${cluster}.bed" \
        --referencePoint center \
        -b 1500 -a 1500 \
        --numberOfProcessors 48 \
        --sortUsing mean \
        -out "08_atac/atac.matrix.${cluster}.gz"

    deeptools plotHeatmap \
        -m "08_atac/atac.matrix.${cluster}.gz" \
        -out "08_atac/atac.matrix.${cluster}.pdf" \
        --sortUsing mean \
        --colorMap  RdYlBu_r RdYlBu_r\
                    RdYlBu_r RdYlBu_r RdYlBu_r RdYlBu_r \
        --samplesLabel  "GFPpos_shCd19" "GFPpos_shRunx3" \
                        "GFPneg_shCd19" "GFPneg_shRunx3" "CX3CR1pos_shCd19" "CX3CR1pos_shRunx3" \
        --regionsLabel "C${cluster}"
done

In [13]:
# Extract accessibility signals

deeptools multiBigwigSummary BED-file -b \
    ${bigWigFiles["GFPpos_shCD19"]} \
    ${bigWigFiles["GFPpos_shRunx3"]} \
    ${bigWigFiles["GFPneg_shCD19"]} \
    ${bigWigFiles["GFPneg_shRunx3"]} \
    ${bigWigFiles["CX3CR1pos_shCD19"]} \
    ${bigWigFiles["CX3CR1pos_shRunx3"]} \
    --BED "01_peakEDA/fullpeak.clean.clustered.named.bed" \
    --outRawCounts 08_atac/atac.signal.tab \
    -o 08_atac/atac.signal.npz

Number of bins found: 4172
